# GiantMed: medication recommendation with SafeDrug and an LLM

This walkthrough explains the GiantMed pipeline implemented in PyHealth. It combines a trained medication model with similar visits, drug interaction evidence and an LLM that revises scores near a decision boundary.

**Audience:** readers familiar with Python and basic machine learning. **You will:** inspect a small synthetic example, understand the input token budget, and read the recorded MIMIC-III demo comparison.

The default notebook is offline. It does **not** call an LLM service or train on patient data. The toy example is fabricated; the final comparison is an aggregate summary recorded from five-visit runs on 13–14 November 2026. The demo is too small to establish clinical utility or reproduce the paper. [Paper](https://doi.org/10.1145/3770854.3780297) · [Reference code](https://github.com/lvhangkenn/GiantMed)

## 1. The idea

1. SafeDrug predicts one probability per medication.
2. GiantMed identifies scores near the recommendation cutoff (the released refinement script uses 0.3 ≤ score < 0.7).
3. It prepares current-visit context, similar **training** visits and interactions among boundary candidates.
4. The LLM proposes revised boundary scores; unmodified scores retain the backbone prediction.

The LLM receives possible interactions as evidence. They are **not** enforced as a hard safety constraint.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Find the checkout when launched either at the repository root or in examples.
working = Path.cwd().resolve()
repo = next((p for p in (working, *working.parents) if (p / "pyhealth").is_dir()), None)
if repo is None:
    raise RuntimeError("Run this notebook from inside a PyHealth checkout.")
import contextlib, io, warnings
with warnings.catch_warnings(), contextlib.redirect_stderr(io.StringIO()):
    warnings.simplefilter("ignore")
    from pyhealth.models import GiantMed
print("PyHealth checkout:", repo.name)

## 2. A small synthetic walkthrough

These codes, visits, interactions and LLM answers are invented. A real model generates its answer; here a deterministic stand-in makes the example easy to run without network access.

In [ ]:
codes = ["A", "B", "C", "D"]
probabilities = np.array([0.91, 0.48, 0.53, 0.12])
ddi = np.zeros((4, 4), dtype=int)
ddi[1, 2] = ddi[2, 1] = 1
training = [{"patient_id": "train-1", "visit_id": "v1",
             "conditions": [["diagnosis-1"]], "procedures": [["procedure-1"]],
             "drugs": ["B"]}]
query = {"patient_id": "demo", "visit_id": "v2",
         "conditions": [["diagnosis-1"]], "procedures": [["procedure-1"]],
         "drugs_hist": [[]]}

def toy_llm(prompt):
    # Fabricated response, not clinical advice or a real model output.
    return '{"B": 0.78, "C": 0.25}'

refiner = GiantMed(codes, ddi, toy_llm, training)
result = refiner.refine_with_details(query, probabilities)
pd.DataFrame({"Drug": codes, "SafeDrug score": probabilities,
              "Refined score": result["probabilities"],
              "Boundary candidate": [c in result["candidates"] for c in codes]})

Only B and C fall in the boundary. In this toy response, B crosses the 0.5 cutoff upward and C crosses it downward; A and D are unchanged. The interaction information is included in the prompt, but a real LLM could still return an interacting combination.

In [ ]:
print("Candidates:", result["candidates"])
print("Accepted:", result["updated"])
print("Response status:", result["status"])
print("Interaction pair present:", '["B", "C"]' in result["prompt"])

## 3. Running the real MIMIC-III demo

The companion script loads the bundled MIMIC-III demo, splits patients, trains SafeDrug for one epoch, builds the retrieval index from training patients, and evaluates held-out visits. The output below **prints a command**; it does not execute it.

The example requires `--ndc-rxcui` and `--ndc-atc` for reference medication labels. Add `--medi` and `--ingredient-cache` to enable MEDI. The [README](giantmed_README.md) explains the required files, licensing and token-budget behavior. See the script for all options. A run using OpenRouter sends clinical and retrieved-visit information to the service and incurs API charges.

In [ ]:
print("Offline smoke test:")
print("python examples/drug_recommendation/giantmed_mimic3.py --ndc-rxcui PATH --ndc-atc PATH --demo-refinement")
print("\nReal LLM (after setting OPENROUTER_API_KEY in your environment):")
print("python examples/drug_recommendation/giantmed_mimic3.py "
      "--openrouter-model qwen/qwen3-14b --prompt-tokenizer Qwen/Qwen3-14B "
      "--medi PATH --ndc-rxcui PATH --ndc-atc PATH --ingredient-cache PATH")
print("\nThe printed commands are examples; no model call is made here.")

## 4. Historical pipeline smoke test: five held-out demo visits

**Historical results use the over-broad PyHealth mapping. Do not compare them directly with the corrected mapping run below or draw performance conclusions.** SafeDrug had only one epoch of training on this tiny demo; 121–123 medications per visit were in the boundary and its baseline recommended about 59 medications per visit. These historical runs used an earlier version of the prompts and budget allocation, and have not been rerun after those changed. The following aggregate numbers are copied from the completed runs, not newly computed by this notebook. All OpenRouter runs used Qwen3-14B with MEDI. The batched run uses four requests per visit and includes cross-batch interactions as context. The long-prompt run exceeded the paper's reported input length; the previous and compact budget runs each kept inputs below 4,096 provider-reported tokens. The compact version uses indexed candidate rows and interactions. Outputs are sampled, so the runs are not a controlled ablation.

In [ ]:
import json
recorded = json.loads(r'''{
  "run_date": "2026-11-13",
  "model": "qwen/qwen3-14b",
  "visits": 5,
  "backbone": "SafeDrug (one epoch, MIMIC-III demo)",
  "baseline": {
    "jaccard_samples": 0.30044472831358077,
    "f1_samples": 0.4511515152367874,
    "pr_auc_samples": 0.48312983821699873,
    "ddi_rate": 0.051713912940339224,
    "mean_medications": 59.25
  },
  "long": {
    "jaccard_samples": 0.21749993080638244,
    "f1_samples": 0.34634980434368234,
    "pr_auc_samples": 0.44429038844796337,
    "ddi_rate": 0.046273589934961085,
    "mean_medications": 51.58333333333333
  },
  "limited": {
    "jaccard_samples": 0.2929214426468708,
    "f1_samples": 0.4407890859748115,
    "pr_auc_samples": 0.43897972670280994,
    "ddi_rate": 0.048738184025669935,
    "mean_medications": 66.75
  },
  "budgets": [
    {
      "prompt_tokens": 3840,
      "candidate_names": [
        57,
        122
      ],
      "history_visits": [
        0,
        0
      ],
      "similar_visits": [
        3,
        3
      ],
      "ddi_pairs": [
        165,
        349
      ]
    },
    {
      "prompt_tokens": 3839,
      "candidate_names": [
        53,
        123
      ],
      "history_visits": [
        1,
        1
      ],
      "similar_visits": [
        1,
        3
      ],
      "ddi_pairs": [
        53,
        314
      ]
    },
    {
      "prompt_tokens": 3839,
      "candidate_names": [
        57,
        122
      ],
      "history_visits": [
        2,
        2
      ],
      "similar_visits": [
        0,
        3
      ],
      "ddi_pairs": [
        18,
        332
      ]
    },
    {
      "prompt_tokens": 3833,
      "candidate_names": [
        60,
        123
      ],
      "history_visits": [
        0,
        0
      ],
      "similar_visits": [
        3,
        3
      ],
      "ddi_pairs": [
        230,
        334
      ]
    },
    {
      "prompt_tokens": 3839,
      "candidate_names": [
        55,
        121
      ],
      "history_visits": [
        0,
        1
      ],
      "similar_visits": [
        2,
        3
      ],
      "ddi_pairs": [
        144,
        320
      ]
    }
  ],
  "provider_prompt_tokens": [
    3840,
    3839,
    3839,
    3833,
    3839
  ],
  "accepted": [
    610,
    611
  ],
  "decision_changes": 54,
  "medi_retained": [
    291,
    644
  ],
  "api_cost_usd": 0.00353562,
  "previous_limited": {
    "jaccard_samples": 0.29503892526280584,
    "f1_samples": 0.4451678055015553,
    "pr_auc_samples": 0.4469561265412141,
    "ddi_rate": 0.04646128226477935,
    "mean_medications": 70.0
  },
  "batched": {
    "jaccard_samples": 0.2895951917091623,
    "f1_samples": 0.4363183009248522,
    "pr_auc_samples": 0.41810943497419734,
    "ddi_rate": 0.046307228125409945,
    "mean_medications": 76.91666666666666
  },
  "batched_budgets": [
    {
      "batch_count": 4,
      "prompt_tokens": [
        2477,
        2952,
        3139,
        2994
      ],
      "candidate_names": [
        122,
        122
      ],
      "resolved_candidate_names": [
        122,
        122
      ],
      "ddi_pairs": [
        349,
        349
      ],
      "cross_batch_pairs": 228,
      "support_per_batch": [
        {
          "history_visits": [
            0,
            0
          ],
          "similar_visits": [
            3,
            3
          ]
        },
        {
          "history_visits": [
            0,
            0
          ],
          "similar_visits": [
            3,
            3
          ]
        },
        {
          "history_visits": [
            0,
            0
          ],
          "similar_visits": [
            3,
            3
          ]
        },
        {
          "history_visits": [
            0,
            0
          ],
          "similar_visits": [
            3,
            3
          ]
        }
      ]
    },
    {
      "batch_count": 4,
      "prompt_tokens": [
        3099,
        3551,
        3739,
        3377
      ],
      "candidate_names": [
        123,
        123
      ],
      "resolved_candidate_names": [
        123,
        123
      ],
      "ddi_pairs": [
        314,
        314
      ],
      "cross_batch_pairs": 202,
      "support_per_batch": [
        {
          "history_visits": [
            1,
            1
          ],
          "similar_visits": [
            3,
            3
          ]
        },
        {
          "history_visits": [
            1,
            1
          ],
          "similar_visits": [
            3,
            3
          ]
        },
        {
          "history_visits": [
            1,
            1
          ],
          "similar_visits": [
            3,
            3
          ]
        },
        {
          "history_visits": [
            1,
            1
          ],
          "similar_visits": [
            3,
            3
          ]
        }
      ]
    },
    {
      "batch_count": 4,
      "prompt_tokens": [
        3512,
        3788,
        3463,
        3582
      ],
      "candidate_names": [
        122,
        122
      ],
      "resolved_candidate_names": [
        122,
        122
      ],
      "ddi_pairs": [
        332,
        332
      ],
      "cross_batch_pairs": 224,
      "support_per_batch": [
        {
          "history_visits": [
            2,
            2
          ],
          "similar_visits": [
            3,
            3
          ]
        },
        {
          "history_visits": [
            2,
            2
          ],
          "similar_visits": [
            3,
            3
          ]
        },
        {
          "history_visits": [
            1,
            2
          ],
          "similar_visits": [
            3,
            3
          ]
        },
        {
          "history_visits": [
            2,
            2
          ],
          "similar_visits": [
            3,
            3
          ]
        }
      ]
    },
    {
      "batch_count": 4,
      "prompt_tokens": [
        2468,
        2764,
        2940,
        2576
      ],
      "candidate_names": [
        123,
        123
      ],
      "resolved_candidate_names": [
        123,
        123
      ],
      "ddi_pairs": [
        334,
        334
      ],
      "cross_batch_pairs": 223,
      "support_per_batch": [
        {
          "history_visits": [
            0,
            0
          ],
          "similar_visits": [
            3,
            3
          ]
        },
        {
          "history_visits": [
            0,
            0
          ],
          "similar_visits": [
            3,
            3
          ]
        },
        {
          "history_visits": [
            0,
            0
          ],
          "similar_visits": [
            3,
            3
          ]
        },
        {
          "history_visits": [
            0,
            0
          ],
          "similar_visits": [
            3,
            3
          ]
        }
      ]
    },
    {
      "batch_count": 4,
      "prompt_tokens": [
        3816,
        3130,
        3244,
        3834
      ],
      "candidate_names": [
        121,
        121
      ],
      "resolved_candidate_names": [
        121,
        121
      ],
      "ddi_pairs": [
        320,
        320
      ],
      "cross_batch_pairs": 214,
      "support_per_batch": [
        {
          "history_visits": [
            1,
            1
          ],
          "similar_visits": [
            0,
            3
          ]
        },
        {
          "history_visits": [
            0,
            1
          ],
          "similar_visits": [
            3,
            3
          ]
        },
        {
          "history_visits": [
            0,
            1
          ],
          "similar_visits": [
            3,
            3
          ]
        },
        {
          "history_visits": [
            1,
            1
          ],
          "similar_visits": [
            0,
            3
          ]
        }
      ]
    }
  ],
  "batched_provider_prompt_tokens": [
    2477,
    2952,
    3139,
    2994,
    3099,
    3551,
    3739,
    3377,
    3512,
    3788,
    3463,
    3582,
    2468,
    2764,
    2940,
    2576,
    3816,
    3130,
    3244,
    3834
  ],
  "batched_accepted": [
    611,
    611
  ],
  "batched_decision_changes": 121,
  "batched_api_cost_usd": 0.0086526
}''')
rows = []
for title, key in [("SafeDrug", "baseline"), ("Qwen3-14B, long prompts", "long"),
                   ("Qwen3-14B, previous budget", "previous_limited"),
                   ("Qwen3-14B, compact budget", "limited"),
                   ("Qwen3-14B, batched", "batched")]:
    value = recorded[key]
    rows.append({"Method": title, "Jaccard ↑": value["jaccard_samples"],
                 "F1 ↑": value["f1_samples"], "PR-AUC ↑": value["pr_auc_samples"],
                 "DDI rate ↓": value["ddi_rate"], "Mean medications": value["mean_medications"]})
comparison = pd.DataFrame(rows).set_index("Method")
comparison.round(3)

**Reading the table:** This is a pipeline smoke test, not a performance comparison. All recorded Qwen3-14B variants had lower Jaccard, F1 and PR-AUC than SafeDrug. Results use five visits and sampled outputs, and compare with recorded medications rather than clinical outcomes.


## 5. What the 4,096-token limit omitted

In the batched run, all current-visit information, candidate codes and scores, candidate names and boundary interaction pairs were retained across requests. Each request includes all interactions involving its target drugs, including cross-batch interactions. Supporting history and similar visits may be omitted. Interactions remain guidance, not a hard safety rule.

In [ ]:
b = recorded["batched_budgets"]
coverage = pd.DataFrame({
    "Visit": range(1, len(b) + 1),
    "Requests": [x["batch_count"] for x in b],
    "Input token range": [f"{min(x['prompt_tokens'])}-{max(x['prompt_tokens'])}" for x in b],
    "Drug names kept": [f"{x['resolved_candidate_names'][0]}/{x['resolved_candidate_names'][1]}" for x in b],
    "Interaction pairs kept": [f"{x['ddi_pairs'][0]}/{x['ddi_pairs'][1]}" for x in b],
})
coverage.set_index("Visit")

In [ ]:
names_kept = sum(x["resolved_candidate_names"][0] for x in b)
names_total = sum(x["resolved_candidate_names"][1] for x in b)
pairs_kept = sum(x["ddi_pairs"][0] for x in b)
pairs_total = sum(x["ddi_pairs"][1] for x in b)
cross_batch = sum(x["cross_batch_pairs"] for x in b)
support = [entry for visit in b for entry in visit["support_per_batch"]]
print(f"Resolved medication names: {names_kept}/{names_total}")
print(f"Unique DDI-pair instances covered: {pairs_kept}/{pairs_total}")
print(f"Cross-batch interaction pairs: {cross_batch}")
print(f"Similar visits retained across requests: {sum(x['similar_visits'][0] for x in support)}/{sum(x['similar_visits'][1] for x in support)}")
print(f"History visits retained across requests: {sum(x['history_visits'][0] for x in support)}/{sum(x['history_visits'][1] for x in support)}")
print(f"Accepted LLM scores: {recorded['batched_accepted'][0]}/{recorded['batched_accepted'][1]}")
print(f"Medication decisions changed: {recorded['batched_decision_changes']}")
print(f"Reported API cost: ${recorded['batched_api_cost_usd']:.4f}")

Batching retained all candidate names and boundary interaction pairs in this demo, including cross-batch pairs. Some history and similar-visit context did not fit every request. Batches are decided independently, so interaction evidence is not a hard constraint. PyHealth's split-aware processor also adds labels seen outside the training split, so a benchmark without that vocabulary exposure needs a fixed medication vocabulary.

## 5b. Historical rerun using PyHealth labels

On 16 November, the MEDI index was changed to use the same NDC-to-ATC3 mapping as the PyHealth task. The following are recorded aggregate results from five demo visits, with Qwen3-14B and the same 32-candidate batch limit. **This is a pipeline smoke test, not evidence of clinical benefit.** The LLM outputs are sampled; the before/after numbers are not a controlled ablation. The one-epoch backbone is weak and the label vocabulary includes held-out labels.

The MEDI ingredient cache remains incomplete. `missing_ingredients` means that a visit/label mapping exists but the supplied RxCUI ingredient cache does not provide ingredients. Such missing evidence is not an indication mismatch; it may still suppress useful retrieval evidence.

**Mapping audit:** One NDC in the cached PyHealth mapping expands to 133 ATC codes across 52 ATC3 labels. Its external RxCUI is a non-current proprietary placeholder, with no reliable ingredient in RxNav. This may inflate the demo vocabulary and distort the results below. A conservative local RxNav cache extension resolved 44 of the original 256 missing ingredient checks; 202 of the 212 still missing involved this NDC. The corrected reference-mapping rerun is reported in the next section; its medication labels and model training differ from this historical run.

In [ ]:
# Recorded aggregates only; no prompts, responses or patient identifiers.
aligned_run = {'date': '2026-11-16', 'visits': 5, 'baseline': {'jaccard_samples': 0.30044472831358077, 'f1_samples': 0.4511515152367874, 'pr_auc_samples': 0.48312983821699873, 'ddi_rate': 0.051713912940339224, 'mean_medications': 59.25}, 'aligned': {'jaccard_samples': 0.2831785041970919, 'f1_samples': 0.43493434832962374, 'pr_auc_samples': 0.44912883397449976, 'ddi_rate': 0.06402867585043576, 'mean_medications': 75.16666666666666}, 'previous': {'jaccard_samples': 0.2668790207951497, 'f1_samples': 0.4165297870299456, 'pr_auc_samples': 0.4545571640297682, 'ddi_rate': 0.06358381502890173, 'mean_medications': 70.58333333333333}, 'medi': {'missing_ingredients': 256, 'indication_match': 91, 'indication_mismatch': 82, 'history_rescue': 215}, 'requests': 20, 'cost': 0.0076538, 'prompt_tokens': [2553, 3824], 'candidate_names': [611, 611], 'ddi_pairs': [1649, 1649], 'similar_visits': [60, 60], 'history_visits': [10, 16]}
rows = []
for title, key in [("SafeDrug", "baseline"), ("GiantMed before alignment", "previous"), ("GiantMed aligned", "aligned")]:
    x = aligned_run[key]
    rows.append({"Method": title, "Jaccard": x["jaccard_samples"], "F1": x["f1_samples"], "PR-AUC": x["pr_auc_samples"], "DDI rate": x["ddi_rate"], "Mean medications": x["mean_medications"]})
pd.DataFrame(rows).set_index("Method").round(4)

In [ ]:
print("MEDI outcomes:", aligned_run["medi"])
print("Candidate names:", aligned_run["candidate_names"])
print("Interaction pairs:", aligned_run["ddi_pairs"])
print("Similar-visit inclusions:", aligned_run["similar_visits"])
print("History inclusions:", aligned_run["history_visits"])
print("Requests / provider input-token range:", aligned_run["requests"], aligned_run["prompt_tokens"])
print("Reported cost (USD):", round(aligned_run["cost"], 6))

## 5c. Five visits with reference medication mapping

The example now uses the reference NDC → RxCUI → first ATC3 mapping for both task labels and MEDI evidence. The problematic NDC has no entry in this chain and is excluded; no substitute label is guessed. This avoids the known over-broad entry but does **not** repair PyHealth’s shared mapping resource. Of 8,920 usable prescription rows in the entire demo, 6,789 mapped; the rest were excluded.

The same five held-out visits were selected, but their medication vocabularies and the trained SafeDrug model differ from the earlier runs. **Do not interpret the tables as a controlled comparison between mapping strategies.** The weak one-epoch backbone, held-out label vocabulary exposure, small cohort and sampled model output make this a pipeline smoke test only. The expanded local RxNav ingredient cache was used; it is not bundled with PyHealth.

MEDI evaluates history rescue first; a rescued medication does not demonstrate ingredient-cache coverage.

In [ ]:
# Recorded aggregate results; no prompts, responses or patient identifiers.
reference_run = {'date': '2026-11-17', 'visits': 5, 'baseline': {'jaccard_samples': 0.25900159841467174, 'f1_samples': 0.40822082608196825, 'pr_auc_samples': 0.49918605283314055, 'ddi_rate': 0.04011194029850746, 'mean_medications': 21.25}, 'giantmed': {'jaccard_samples': 0.24820716320716318, 'f1_samples': 0.3936324206246823, 'pr_auc_samples': 0.3836284914071797, 'ddi_rate': 0.09941520467836257, 'mean_medications': 29.166666666666664}, 'medi': {'indication_mismatch': 106, 'indication_match': 102, 'history_rescue': 83}, 'requests': 15, 'cost': 0.00548486, 'prompt_tokens': [2097, 3424], 'candidate_names': [394, 394], 'ddi_pairs': [1383, 1383], 'cross_batch_pairs': 817, 'similar_visits': [45, 45], 'history_visits': [12, 12], 'prescription_ndc_rows': [6789, 8920]}
rows = []
for title, key in [("SafeDrug", "baseline"), ("GiantMed, reference mapping", "giantmed")]:
    x = reference_run[key]
    rows.append({"Method": title, "Jaccard": x["jaccard_samples"], "F1": x["f1_samples"], "PR-AUC": x["pr_auc_samples"], "DDI rate": x["ddi_rate"], "Mean medications": x["mean_medications"]})
pd.DataFrame(rows).set_index("Method").round(4)

In [ ]:
print("Usable prescription rows mapped:", reference_run["prescription_ndc_rows"])
print("MEDI outcomes:", reference_run["medi"])
print("Candidate names:", reference_run["candidate_names"])
print("Boundary interaction pairs:", reference_run["ddi_pairs"])
print("Similar-visit inclusions:", reference_run["similar_visits"])
print("History inclusions:", reference_run["history_visits"])
print("Requests / provider input-token range:", reference_run["requests"], reference_run["prompt_tokens"])
print("Reported cost (USD):", round(reference_run["cost"], 6))

## 5d. Training SafeDrug for ten epochs

On 19 November, the same five visits, reference medication mapping, split, Qwen3-14B model and MEDI resources were used with a freshly trained SafeDrug backbone. The run evaluated the **final tenth epoch**; no early stopping or validation-based checkpoint selection was enabled. The validation set has only two samples. These sampled results illustrate the pipeline and cannot establish clinical benefit.

Ten-epoch GiantMed improved Jaccard and F1 over its own backbone on these visits, while PR-AUC and DDI rate were worse. The backbone itself also changed as it trained longer. The task's label vocabulary can include held-out labels and unmapped prescription rows are excluded.

In [ ]:
# Recorded aggregates only; no patient prompts, responses or identifiers.
epochs10 = {'date': '2026-11-19', 'epochs': 10, 'visits': 5, 'baseline': {'jaccard_samples': 0.31937115857655585, 'f1_samples': 0.48228005358223147, 'pr_auc_samples': 0.615668007217317, 'ddi_rate': 0.08528784648187633, 'mean_medications': 14.166666666666668}, 'giantmed': {'jaccard_samples': 0.335519033976257, 'f1_samples': 0.500042376104532, 'pr_auc_samples': 0.6108673733360401, 'ddi_rate': 0.10182767624020887, 'mean_medications': 18.083333333333336}, 'decision_changes': {'added_recorded': 9, 'added_unrecorded': 23, 'removed_recorded': 3, 'removed_unrecorded': 10}, 'previous_decision_changes': {'added_recorded': 14, 'added_unrecorded': 74, 'removed_recorded': 9, 'removed_unrecorded': 39}, 'previous_candidates': 394, 'current_candidates': 99, 'ddi_pairs': [95, 95], 'requests': 5, 'cost': 0.00099198, 'prompt_tokens': [1074, 1872], 'validation_pr_auc': [0.4438, 0.4568, 0.5193, 0.5132, 0.5494, 0.5485, 0.5722, 0.5794, 0.599, 0.5776]}
rows = []
for title, key in [("SafeDrug, 10 epochs", "baseline"), ("GiantMed, 10 epochs", "giantmed")]:
    x = epochs10[key]
    rows.append({"Method": title, "Jaccard": x["jaccard_samples"], "F1": x["f1_samples"], "PR-AUC": x["pr_auc_samples"], "DDI rate": x["ddi_rate"], "Mean medications": x["mean_medications"]})
pd.DataFrame(rows).set_index("Method").round(4)

The boundary set decreased from 394 to 99 candidates. Counts below are pooled visit–medication decisions, while Jaccard and F1 are averaged within patients and then across patients. “Recorded” means present in the medication labels, not necessarily clinically appropriate. The prior one-epoch response counts were obtained by replaying the saved responses locally.

In [ ]:
print("One-epoch changes:", epochs10["previous_decision_changes"])
print("Ten-epoch changes:", epochs10["decision_changes"])
print("Boundary candidates, one vs ten epochs:", epochs10["previous_candidates"], epochs10["current_candidates"])
print("Validation PR-AUC by epoch:", epochs10["validation_pr_auc"])
print("Requests, prompt-token range and cost:", epochs10["requests"], epochs10["prompt_tokens"], round(epochs10["cost"], 6))

## 6. Try it yourself

**Exercise:** In the synthetic example, change B's proposed score to `0.45`. Would B be recommended at the 0.5 threshold? The cell below gives the answer without making a network request.

In [ ]:
exercise_score = 0.45
print("B recommended:", exercise_score >= 0.5)

**Answer:** No. `0.45` is below the 0.5 cutoff.

**Common pitfall:** passing a DDI matrix or medication codes in a different order from the backbone's output can produce incorrect evidence. Keep the matrix, codes and prediction vector aligned. For a fuller experiment, compare the same patient split and preprocessing under multiple seeds and document all omitted evidence and provider token counts.

## 7. Try a prompt policy and local run record

The instruction template is a **patient-independent, single-line** string containing `{boundary}`. GiantMed still constructs the evidence sections, enforces the token budget and defines the JSON response format. Custom instructions can conflict with the format or change behavior, so compare them on validation data. The code below uses only invented data and writes an aggregate manifest in a temporary local directory; no LLM, MLflow service or patient prompt is sent anywhere.

To opt in to MLflow for the demo, provide `--tracking mlflow --tracking-uri URI`. Add `--tracking-fallback --tracking-dir PATH` only if a local fallback is desired. Without that option, MLflow failures raise an error. The manifest records the exact template and its SHA-256, renderer version, aggregate metrics and supplied settings. It does not automatically record rendered prompts, responses or per-epoch metrics. Never put patient data or secrets into a template or tracking metadata.

In [ ]:
import json
import tempfile
from pathlib import Path
from pyhealth.experiment_tracking import record_experiment

custom = GiantMed(
    ["A"], [[0]], lambda prompt: '{"A": 0.8}', [],
    prompt_template="Assess {boundary} conservatively and consider interactions.",
)
result = custom.refine_with_details({"patient_id": "synthetic", "drugs": ["SECRET_TARGET"]}, [0.5])
with tempfile.TemporaryDirectory() as directory:
    saved = record_experiment(
        params={"epochs": 1}, metrics={"accepted_scores": len(result["updated"])},
        prompt=custom.prompt_metadata(), local_dir=directory,
    )
    manifest = json.loads(Path(saved["path"]).read_text())
    print("Backend and status:", saved["backend"], manifest["tracking"]["status"])
    print("Template SHA-256:", manifest["prompt"]["template_sha256"])
    print("Patient target in manifest:", "SECRET_TARGET" in json.dumps(manifest))